# Experiment 4 — Finetune the untrained model directly on `dataset_matan`

Warm-starts from **`cyttic/trocr-hebrew-untrained`** (ViT-handwritten encoder +
DictaBERT decoder, **randomly-initialized cross-attention** — see the repo
`CLAUDE.md`) and finetunes it directly on **`cyttic/trocr-hebrew-matan`**
(real handwritten Hebrew lines, writer-level train/test split) with the
**encoder unfrozen** for the whole run.

| Setting | Value |
|---|---|
| Warm-start model | `cyttic/trocr-hebrew-untrained` |
| Dataset | `cyttic/trocr-hebrew-matan` (`train` / `test`, writer-level split) |
| Encoder | **unfrozen** (full model trains from step 1) |
| Epochs | **8** |
| Batch size | **16** |
| Precision | **bf16** (target hardware: L4, 24 GB) |
| Checkpoints -> Hub | **once per epoch** (`save_strategy="epoch"`, `hub_strategy="checkpoint"`) |
| Final model | pushed to `cyttic/trocr-hebrew-matan-finetuned` |

**Baseline first:** before training starts, the notebook runs a full beam-search
CER/WER/BLEU pass with the *untrained* weights on the matan test split, so you
have a "before" number to compare the finetuned ("after") result against —
the same metrics, computed the same way, both times.

> Run top to bottom on an L4. Checkpoints land on the Hub every epoch, so a lost
> Colab session can resume — just re-run top to bottom.

## 1. Setup

Self-contained: downloads the model + dataset from HuggingFace. Run top to bottom.

In [ ]:
# Run once on a fresh VM/Colab runtime. Comment out if deps are already installed.
!pip install -q torch transformers datasets accelerate jiwer sacrebleu pillow matplotlib

In [ ]:
import os
import torch
import jiwer
import sacrebleu
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    VisionEncoderDecoderModel,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("gpu   :", torch.cuda.get_device_name(0))
    print("vram  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
    print("bf16  :", torch.cuda.is_bf16_supported())

In [ ]:
# --- HuggingFace auth (Colab has no persistent box auth -> log in every session) ---
# Needs a WRITE token. Create one at https://huggingface.co/settings/tokens (token type: Write)
from huggingface_hub import notebook_login, whoami
notebook_login()
print("logged in as:", whoami()["name"])

In [ ]:
# --- HebrewBlockProcessor (inlined so this notebook is self-contained) ---
from PIL import Image, ImageOps

class HebrewBlockProcessor:
    """Mirror (RTL->LTR) -> resize to 64px height -> tile into a 384x384 ViT container."""
    TARGET_HEIGHT = 64
    CONTAINER_SIZE = 384
    IMAGE_MEAN = [0.5, 0.5, 0.5]
    IMAGE_STD = [0.5, 0.5, 0.5]

    def __call__(self, images, return_tensors="pt"):
        if not isinstance(images, list):
            images = [images]
        pixel_values = torch.stack([self._process(img) for img in images])
        return {"pixel_values": pixel_values}

    def _process(self, image):
        image = image.convert("RGB")
        image = ImageOps.mirror(image)
        w, h = image.size
        new_w = max(1, round(w * self.TARGET_HEIGHT / h))
        image = image.resize((new_w, self.TARGET_HEIGHT), Image.LANCZOS)
        container = Image.new("RGB", (self.CONTAINER_SIZE, self.CONTAINER_SIZE), (255, 255, 255))
        img_arr = np.array(image)
        src_x, dest_x, dest_y = 0, 0, 0
        while src_x < new_w and dest_y < self.CONTAINER_SIZE:
            chunk_w = min(new_w - src_x, self.CONTAINER_SIZE - dest_x)
            chunk = Image.fromarray(img_arr[:, src_x:src_x + chunk_w])
            container.paste(chunk, (dest_x, dest_y))
            src_x += chunk_w
            dest_x += chunk_w
            if dest_x >= self.CONTAINER_SIZE:
                dest_x = 0
                dest_y += self.TARGET_HEIGHT
        t = torch.tensor(np.array(container), dtype=torch.float32).permute(2, 0, 1) / 255.0
        mean = torch.tensor(self.IMAGE_MEAN).view(3, 1, 1)
        std = torch.tensor(self.IMAGE_STD).view(3, 1, 1)
        return (t - mean) / std

## 2. Config

Fixed for this experiment: `ENCODER_FROZEN = False` (unfrozen from the start —
the cross-attention is random-init and the encoder needs to adapt to real ink),
`EPOCHS = 8`, `BATCH_SIZE = 16`, `PRECISION = "bf16"` (L4).

In [ ]:
# --- warm-start from the UNTRAINED base, finetune on the matan (real handwriting) set ---
MODEL_ID          = "cyttic/trocr-hebrew-untrained"   # ViT-handwritten encoder + DictaBERT decoder, random cross-attn
DATASET_ID        = "cyttic/trocr-hebrew-matan"       # writer-level train/test split (see to_parquet_pairs.py)

EPOCHS            = 8
BATCH_SIZE        = 16
GRAD_ACCUM        = 1
LR                = 5e-5     # matches train.py's default for finetuning the untrained model directly
MAX_TARGET_LENGTH = 128
NUM_WORKERS       = 4
PRECISION         = "bf16"   # target hardware: L4 (24GB, bf16). Use "fp16" on a Turing card (RTX 2080).
MAX_STEPS         = -1       # set e.g. 50 for a quick smoke test

# --- encoder freeze ---
# This experiment's whole point: the encoder trains from the first step, alongside
# the (randomly-initialized) cross-attention and the decoder -- nothing stays fixed.
ENCODER_FROZEN    = False

RUN_NAME       = "trocr-hebrew-matan-finetuned"
OUTPUT_DIR     = f"output/{RUN_NAME}"          # local staging on the (ephemeral) Colab VM
HUB_CKPTS_REPO = f"cyttic/{RUN_NAME}-ckpts"    # durable: latest checkpoint pushed here once per epoch
HUB_REPO       = f"cyttic/{RUN_NAME}"          # final model repo

print("warm-start   :", MODEL_ID)
print("dataset      :", DATASET_ID)
print("variant      :", "frozen encoder" if ENCODER_FROZEN else "FULL (encoder unfrozen)")
print("epochs       :", EPOCHS, "| lr:", LR, "| batch:", BATCH_SIZE, "| precision:", PRECISION)
print("local staging:", OUTPUT_DIR)
print("hub ckpts    :", HUB_CKPTS_REPO)
print("hub final    :", HUB_REPO)

SAVE_LIMIT = 3   # keep only the last N local checkpoints (Hub keeps just the latest via hub_strategy="checkpoint")

## 3. Model, tokenizer, processor

Loaded fresh from the **untrained** base. With `ENCODER_FROZEN = False` the
trainable parameter count equals the total — the whole model (encoder, decoder,
and the random cross-attention) trains together.

In [ ]:
model     = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
processor = HebrewBlockProcessor()

# generation_config takes priority over model.config in recent transformers,
# so set the special tokens on it explicitly or generate() fails during eval.
model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.sep_token_id
model.generation_config.max_new_tokens = None

if ENCODER_FROZEN:
    for p in model.encoder.parameters():
        p.requires_grad = False

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"params total    : {n_total/1e6:.1f}M")
print(f"params trainable: {n_train/1e6:.1f}M")

## 4. Dataset

In [ ]:
ds = load_dataset(DATASET_ID)
print(ds)

eval_ds = ds["test"]
print("eval (test) samples:", len(eval_ds))

## 5. Sanity checks

`dataset_matan` carries a `writer` column (see `to_parquet_pairs.py`) so we can
confirm the split is **writer-level / leak-free** — no writer's lines appear in
both `train` and `test` (the same principle `CLAUDE.md` requires for the human
set: a per-line split would let the model "recognize the writer" rather than
read the text). Then *see* what `HebrewBlockProcessor` does to a real line:
mirror (RTL->LTR) -> 64px -> tile into 384x384.

In [ ]:
tr_writers = set(ds["train"]["writer"])
te_writers = set(ds["test"]["writer"])
print(f"train writers: {len(tr_writers)} | test writers: {len(te_writers)} | "
      f"OVERLAP: {len(tr_writers & te_writers)} (should be 0)")

In [ ]:
sample = ds["train"][0]
img = sample["image"].convert("RGB")
print("writer:", sample["writer"], "| text:", sample["text"])

pv = processor([img])["pixel_values"][0]
shown = (pv * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].imshow(img);   ax[0].set_title("raw matan line"); ax[0].axis("off")
ax[1].imshow(shown); ax[1].set_title("after HebrewBlockProcessor (mirrored + tiled 384x384)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 6. Baseline CER / WER / BLEU — BEFORE finetuning

Runs `eval_cer_wer_bleu()` on a 200-line subset of the matan **test** split with
the *untrained* weights and reports CER/WER/BLEU. This is the "before" number.

> Greedy decoding + a short `max_length` here, **on purpose**: the untrained
> model's cross-attention is random, so it almost never predicts EOS, and beam
> search would force every one of the 904 test lines out to the full 128-token
> cap (effectively hangs). The numbers will look like noise either way — that's
> what a baseline for an untrained model looks like. Section 9 reruns the exact
> same helper on the *finetuned* model — full test split, beam=4, max_length=128 —
> for the real "before vs after" comparison.

> `trainer.evaluate()` only reports CER/WER (greedy, by default), so BLEU needs
> its own generation pass — `eval_cer_wer_bleu()` below does CER+WER+BLEU
> together and is reused for both the baseline and the final evaluation.

In [ ]:
@torch.no_grad()
def eval_cer_wer_bleu(model, dataset, beams=4, batch_size=BATCH_SIZE, max_length=MAX_TARGET_LENGTH, tag=""):
    model.eval()
    refs, hyps = [], []
    for start in range(0, len(dataset), batch_size):
        batch = dataset[start:start + batch_size]
        imgs = [im.convert("RGB") for im in batch["image"]]
        pv = processor(imgs)["pixel_values"].to(model.device, dtype=model.dtype)
        ids = model.generate(pv, num_beams=beams, max_length=max_length)
        hyps.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
        refs.extend(batch["text"])
        done = min(start + batch_size, len(dataset))
        if done % (batch_size * 5) == 0 or done == len(dataset):
            print(f"  {tag} {done}/{len(dataset)}", flush=True)

    cer = jiwer.cer(refs, hyps)
    wer = jiwer.wer(refs, hyps)
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    exact = sum(r.strip() == h.strip() for r, h in zip(refs, hyps)) / len(refs)
    return {"cer": cer, "wer": wer, "bleu": bleu, "exact": exact, "refs": refs, "hyps": hyps}


# The UNTRAINED model has random cross-attention -> it almost never predicts EOS,
# so beam search would run every sample out to the full max_length (4 beams x 128
# steps x 904 lines = effectively hangs). For a "before" baseline, greedy decoding +
# a short cap + a representative subset is plenty -- the numbers will be near-garbage
# either way (that's the whole point of a baseline), and Section 9 reruns the SAME
# routine on the finetuned model (which terminates normally) for the real comparison.
BASELINE_SUBSET = min(200, len(eval_ds))
baseline_ds = eval_ds.select(range(BASELINE_SUBSET))

print(f"Running BASELINE eval (untrained weights) on {BASELINE_SUBSET} matan test lines, "
      f"greedy, max_length=64 ...")
baseline = eval_cer_wer_bleu(model, baseline_ds, beams=1, max_length=64, tag="[baseline]")
print(f"\nBASELINE (before finetuning)  |  N={BASELINE_SUBSET}  |  "
      f"CER {baseline['cer']*100:.2f}%  WER {baseline['wer']*100:.2f}%  "
      f"BLEU {baseline['bleu']:.2f}  |  exact {baseline['exact']*100:.2f}%")

## 7. Collator + metrics (CER / WER, used by the Trainer during training)

In [ ]:
def collate(batch):
    images = [ex["image"].convert("RGB") for ex in batch]
    texts  = [ex["text"] for ex in batch]
    pixel_values = processor(images)["pixel_values"]
    labels = tokenizer(
        texts, padding="longest", truncation=True,
        max_length=MAX_TARGET_LENGTH, return_tensors="pt",
    ).input_ids
    labels[labels == tokenizer.pad_token_id] = -100
    return {"pixel_values": pixel_values, "labels": labels}


def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    pred_ids  = np.where(pred_ids  < 0, tokenizer.pad_token_id, pred_ids)
    label_ids = np.where(label_ids < 0, tokenizer.pad_token_id, label_ids)
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"cer": jiwer.cer(label_str, pred_str),
            "wer": jiwer.wer(label_str, pred_str)}

## 8. Train (checkpoints to the Hub ONCE PER EPOCH, auto-resume)

`save_strategy="epoch"` + `hub_strategy="checkpoint"` -> the model is checkpointed
and pushed to `HUB_CKPTS_REPO` **only at the end of each epoch** (not by step
count). `eval_strategy="epoch"` keeps the in-training CER/WER readout aligned
with those same checkpoints.

**If the session dies, just re-run top to bottom** — the train cell looks for
the last checkpoint locally, then on the Hub, and resumes from there.

In [ ]:
targs = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    bf16=(PRECISION == "bf16"),
    fp16=(PRECISION == "fp16"),
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=1,
    eval_strategy="epoch",
    save_strategy="epoch",        # <- checkpoint (and push to the Hub) ONLY at the end of every epoch
    logging_steps=50,
    save_total_limit=SAVE_LIMIT,
    load_best_model_at_end=False, # keep the latest epoch's weights, not "best by eval CER"
    dataloader_num_workers=NUM_WORKERS,
    remove_unused_columns=False,  # keep image/text for the custom collator
    report_to="none",
    push_to_hub=True,
    hub_model_id=HUB_CKPTS_REPO,
    hub_strategy="checkpoint",    # only the latest checkpoint kept on the Hub (for resume)
    hub_private_repo=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=targs,
    train_dataset=ds["train"],
    eval_dataset=eval_ds,
    data_collator=collate,
    compute_metrics=compute_metrics,
)

In [ ]:
# Resume after a lost session: Colab has no durable local disk, so the latest
# checkpoint lives on the Hub (hub_strategy="checkpoint" -> a "last-checkpoint" subfolder).
from huggingface_hub import snapshot_download
from huggingface_hub.utils import RepositoryNotFoundError

last_ckpt = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
if last_ckpt is None:
    try:
        snapshot_download(HUB_CKPTS_REPO, repo_type="model",
                          local_dir=OUTPUT_DIR, allow_patterns="last-checkpoint/*")
        cand = os.path.join(OUTPUT_DIR, "last-checkpoint")
        last_ckpt = cand if os.path.isdir(cand) else None
    except RepositoryNotFoundError:
        pass  # first run -- ckpts repo does not exist yet

print("Resuming from", last_ckpt) if last_ckpt else print("No checkpoint -- training from the untrained warm-start weights")
trainer.train(resume_from_checkpoint=last_ckpt)

## 9. Final CER / WER / BLEU — AFTER finetuning

Reruns the **exact same** `eval_cer_wer_bleu()` routine from Section 6 (same
beam size, same test split, same metrics) so the "before" and "after" numbers
are directly comparable.

In [ ]:
print(f"Running FINAL eval (finetuned weights) on {len(eval_ds)} matan test lines, beam=4 ...")
final = eval_cer_wer_bleu(model, eval_ds, beams=4, tag="[final]")
print(f"\nFINAL (after {EPOCHS} epochs)  |  N={len(eval_ds)}  |  "
      f"CER {final['cer']*100:.2f}%  WER {final['wer']*100:.2f}%  "
      f"BLEU {final['bleu']:.2f}  |  exact {final['exact']*100:.2f}%")

print("\n================  BEFORE vs AFTER  ================")
print(f"  CER   : {baseline['cer']*100:6.2f}%  ->  {final['cer']*100:6.2f}%")
print(f"  WER   : {baseline['wer']*100:6.2f}%  ->  {final['wer']*100:6.2f}%")
print(f"  BLEU  : {baseline['bleu']:6.2f}   ->  {final['bleu']:6.2f}")
print(f"  exact : {baseline['exact']*100:6.2f}%  ->  {final['exact']*100:6.2f}%")
print("====================================================")

## 10. Look at predictions

In [ ]:
model.eval()
n = 6
fig, axes = plt.subplots(n, 1, figsize=(10, 2.2 * n))
for ax, ex in zip(axes, ds["test"].select(range(n))):
    img = ex["image"].convert("RGB")
    pv = processor([img])["pixel_values"].to(model.device, dtype=model.dtype)
    with torch.no_grad():
        ids = model.generate(pv, num_beams=4, max_new_tokens=MAX_TARGET_LENGTH)
    pred = tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"GT  : {ex['text']}\nPRED: {pred}", loc="left", fontsize=9)
plt.tight_layout(); plt.show()

## 11. Save & push the model

Saves locally and pushes the final model to `cyttic/trocr-hebrew-matan-finetuned`.

In [ ]:
final_dir = f"{OUTPUT_DIR}/final"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print("saved ->", final_dir)

# Durable copy so you can warm-start / evaluate from any machine.
model.push_to_hub(HUB_REPO)
tokenizer.push_to_hub(HUB_REPO)
print("pushed ->", HUB_REPO)